## GPT prompting: example notebook with all prompts

### requires python >= 3.10

## Working with batch mode:

#### Parts 1 (preparation) and 2 (gpt client) have to be run every time the notebook is opened.

#### Part 3 is required if new batch file has to be created

#### Part 4 can be run in parts

    The user can submit the batch job, close the notebook (if need be) and return later (then parts 1 and 2 have to be run again). If this notebook is duplicated then multiple datasets can be processed in parallel (be mindful of the job_id-s and that correct job files are used).

**Since the prompts work in a chain (previous prompt's output is the next prompt's input) then the batch jobs for other prompts should not be created or submitted before previous prompt data has been retrieved and added to the result file.**

If the user wants, then the prompts can all have the same exact data as input without other prompt results (requires changes in code). In that case, multiple batch jobs can be submitted in parallel. That would increase the cost of prompting as the size of the input would not decrease with each prompt.


Code for batch prompting is from here: https://developers.openai.com/cookbook/examples/batch_processing.

Tutorial: https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/batch?tabs=global-batch%2Cstandard-input%2Cpython-key&pivots=ai-foundry-portal

The portal for submitting batch jobs: https://ai.azure.com/

In [2]:
#!pip install --upgrade openai

In [1]:
# for auto-reloading extenrnal modules
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

#%reload_ext autoreload

In [2]:
import pandas as pd
from tqdm import tqdm
import openai 
import os
from openai import AzureOpenAI
import configparser
import json
import csv
import sys
from pathlib import Path
import requests

In [3]:
#sys.path.append("../../../")
sys.path.append("../")
from common_code.gpt_utils import *
from common_code.gpt_reply_formats import *

In [4]:

from prompts.semantic_categories.v01.prompt import (
    ALIVE_SYSTEM_PROMPT, ALIVE_FEW_SHOTS_STR, ALIVE_FEW_SHOTS,
    EVENT_SYSTEM_PROMPT, EVENT_FEW_SHOTS_STR, EVENT_FEW_SHOTS,
    TIME_SYSTEM_PROMPT, TIME_FEW_SHOTS_STR, TIME_FEW_SHOTS,
    LOC_SYSTEM_PROMPT, LOC_FEW_SHOTS_STR, LOC_FEW_SHOTS,
    ABSTRACT_SYSTEM_PROMPT, ABSTRACT_FEW_SHOTS_STR, ABSTRACT_FEW_SHOTS,
    STATE_SYSTEM_PROMPT, STATE_FEW_SHOTS_STR, STATE_FEW_SHOTS,
)

In [5]:
pd.set_option('display.max_colwidth', None)
pd.set_option("display.show_dimensions", True)

# 1. Preparation

In [6]:
INPUT_FOLDER = "../data/datasets_500_all_results/" 

OUTPUT_FOLDER = "../data/datasets_500_all_results/" 


INPUT_FILE = os.path.join(INPUT_FOLDER, "ELT_n80_500.csv")

# output file always has to have the dataset class (ELT, A, S etc) as the first thing in filename (needed to extract dataset tag)
OUTPUT_FILE = os.path.join(OUTPUT_FOLDER, "ELT_n80_500_batch_tagged_final.csv")

AI_CONF_FILE = "../../../v04_verb-case_pattern/minu_code/azure_gb.ini"

INPUT_FILE_BASE = Path(INPUT_FILE).stem

DATASET_TAG = Path(OUTPUT_FILE).stem.split("_")[0]

BS = 30


In [7]:
def get_rows(df, decision_columns):
    """Get data rows for gpt. Based on columns that have to be None/'no'."""
    mask = pd.Series(True, index=df.index)

    for col in decision_columns:
        if col in ["A", "S", "T", "L", "L1", "L2", "E"]:
            mask &= df[col] == "no"
        elif col in ["ner_tag", "timex_tag"]:
            mask &= df[col].isna()

    filtered = df[mask]

    return filtered

In [8]:
def merge_data(df1_base, df_answ, tag):
    key_cols = ['sentence_id','head_id', 'head_loc', "verb", "verb_compound", "morph_case", "form"]

    df_selected = df_answ[key_cols + [tag]].copy()

    df1_base['verb_compound'] = df1_base['verb_compound'].astype('string').str.strip()
    df_selected['verb_compound'] = df_selected['verb_compound'].astype('string').str.strip()

    # Merge df1 with df2_selected etc
    merged_df = df1_base.merge(df_selected, on=key_cols, how='left')

    merged_df[tag] = merged_df[tag].fillna("no")
    
    return merged_df

# 2. GPT client

## GPT jaoks vajalik

In [9]:
config = configparser.ConfigParser()

status = config.read(AI_CONF_FILE) 
assert status == [AI_CONF_FILE]

DEPLOYMENT = config['azure-configuration']['deployment_id']

# 3. Functions for batch mode

In [10]:
def make_task(idx, my_batch, few_shots, system_prompt, deployment):
    """Make single task"""
    
    structured_few_shots = []

    i = 0
    while i < len(few_shots):
        user_msg = few_shots[i]
        assistant_msg = few_shots[i + 1] if i + 1 < len(few_shots) else None

        if assistant_msg is None:
            break

        structured_few_shots.append({
            "l": user_msg["content"]["l"],
            "c": user_msg["content"]["c"],
            "a": assistant_msg["content"]["a"],
            "r": assistant_msg["content"].get("r", "")
        })
        i += 2
    
    user_payload = {
            "few_shots": structured_few_shots, #few_shots,
            "batch": my_batch
        }

    task = {
        "custom_id": f"task-{idx}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            # This is what you would have in your Chat Completions API call
            "model": deployment,
            "temperature": 0,
            "response_format": { 
                "type": "json_object"
            },
            "messages": [
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": json.dumps(user_payload, ensure_ascii=False)
                }
            ],
        }
    }

    return task



# Creating an array of json tasks
def task_array(df, few_shots, system_prompt, deployment):
    """Make an array of tasks."""
    tasks = []

    task_batch = []

    rows = df.to_dict(orient="records")

    index = 0
    for df_batch in tqdm(chunk_data(rows, size=BS)):

        batch = []
        batch_data = []
        for k, ex in enumerate(df_batch):
            #batch.append( json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))
            batch.append( {"id": k, "l": ex["sentence"], "c": ex["form"]} )
            batch_data.append((ex["sentence_id"], ex["head_id"], ex["head_loc"],
                              ex["verb"], ex["verb_compound"], ex["morph_case"], 
                              ex["sentence"], ex["form"]))

        task = make_task(index, batch, few_shots, system_prompt, deployment)
        task_batch.append((task["custom_id"], batch_data))

        tasks.append(task)
        index += 1
        
    return tasks, task_batch

# 4. Prompting

# ALIVE -> EVENT -> TIME -> LOC -> STATE

## If ner and timex tagging has to be validated then ner_tag and timex_tag has to be removed from every FILTER_COLS variable

## ALIVE

In [11]:
# algne andmefail
df1 = pd.read_csv(INPUT_FILE, encoding="utf-8",  sep=",")

FILTER_COLS = [] #"ner_tag,timex_tag".split(",")
df2 = get_rows(df1, FILTER_COLS)

df = df2.copy() #.iloc[:30]
#df = df.sample(frac=1)

prompt_tag = "A"

In [12]:
df

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,tags,timex_tag,ekilex_tag,ner_tag
0,449951,704444,18,otsima,NaN,el,detsember,detsembrist,"Lisaks peeti samas kinni sel hetkel sõiduautoga koormat turvanud venelane , keda keskkriminaalpolitsei otsis alates eelmise aasta detsembrist taga seoses narkokuriteoga .",|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,DATE,time,NaN
1,11389924,18251429,3,uskuma,NaN,ad,näide,näitel,Usub enda näitel väikeriigi võimalustesse,NaN,NaN,NaN,NaN
2,15758867,24580070,13,täitma,NaN,ad,kinnitamine,kinnitamisel,"Ülesanded jagatakse nii , et iga ametnik täidab EAGGF arvele kantavate kulutuste kinnitamisel , väljamaksmisel või arvestamisel ainult ühte funktsiooni ning et iga ametniku tööd valvaks mõni teine ametnik .",NaN,NaN,NaN,NaN
3,12182310,19504428,12,jälgima,NaN,abl,internetilehekülg,internetileheküljelt,"Et teleülekannet ei olnud siin võimalik näha , jälgisin seisu Livescore'i internetileheküljelt .",NaN,NaN,NaN,NaN
4,53706,93122,7,vajama,NaN,in,keha,kehas,"“ Ma ei vaja plasti oma kehas selleks , et end naisena tunda ja end teistelegi naisena tutvustada .",NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
245506,5237903,8403209,19,märkima,NaN,in,keskus,Keskuses,""" Kõrge tööhõive ilma= sotsiaalpoliitikata on ohtlik ja ebaõiglane , "" märkis Padrig Flynn Euroopa Poliitika Uuringute Keskuses .",NaN,NaN,NaN,NaN
245507,2917860,4677736,14,pidama,maha,in,raadio,raadios,"Kui EHI võttis Eestis esimesena kasutusele ainepunktisüsteemi , pidasid lugupeetavad kolleegid Tartu ülikoolist raadios maha mitu pikka kõnetundi , kus nad selgitasid , et see süsteem ei hakka Eestis kunagi tööle .",NaN,NaN,NaN,NaN
245508,3196489,5133971,17,tormama,välja,el,koolimaja,koolimajast,"Ligi 53 tundi kestnud pantvangidraama Põhja-Osseetias Beslanis jõudis verise lahenduseni eile , kui peaaegu üheaegselt tormas koolimajast välja rühm pantvange , puhkes tulistamine ning Vene eriüksused läksid vabastusrünnakule .",|L|AL|EL|LS|LT|ALT|ELT|LST|,NaN,location,NaN
245509,14936649,23449513,20,puhastama,NaN,ad,ala,alal,Tallinna Vesi puhastab joogivee kvaliteedi parandamiseks veetorustikke 21. veebruaril kella 24–06 Tammsaare tee ja Laki 26 vahelisel alal .,NaN,NaN,NaN,NaN


In [13]:
# make task array
tasks, task_batch = task_array(df, ALIVE_FEW_SHOTS, ALIVE_SYSTEM_PROMPT, DEPLOYMENT)

8184it [00:00, 11524.83it/s]


In [14]:
# save task array and extra info to files

file_name = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_tasks.jsonl")

with open(file_name, 'w') as file:
    for obj in tasks:
        file.write(json.dumps(obj,ensure_ascii=False) + '\n')
        
f2 = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_data.jsonl")

with open(f2, 'w') as file:
    for obj in task_batch:
        file.write(json.dumps(obj,ensure_ascii=False) + '\n')

#### Retrieving results

In [15]:
result_file_name = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_tasks_results.jsonl")

In [16]:
# Loading data from saved file
results = []
with open(result_file_name, 'r') as file:
    for line in file:
        # Parsing the JSON string into a dict and appending to the list of results
        json_object = json.loads(line.strip())
        results.append(json_object)

In [17]:
len(results)

8184

In [18]:
merged_data = []

for res in results:
    task_id = res['custom_id']
    index = task_id.split('-')[-1]
    result = res['response']['body']['choices'][0]['message']['content']
    answers = json.loads(result)["results"]

    task_data = None
    for row in task_batch:
        if row[0] == task_id:
            task_data = row
            break
    
    if len(answers) == len(task_data[1]):
        c = 0
        for task, answer in zip(task_data[1], answers):
            if c == answer["id"]:
                d = task + (answer["a"],)
                merged_data.append(d)
                c+= 1
            else:
                print(f"missing an answer for task {task_id} id {c}")
                d = task + ("",)
                merged_data.append(d)
    else:
        print(f"{task_id}:", (len(answers)), "vs", (len(task_data[1])))
        
    if task_data is None:
        print("No task_data for task_id", task_id)
        
cols = ["sentence_id","head_id","head_loc", "verb","verb_compound","morph_case","sentence","form", prompt_tag]
mdf = pd.DataFrame(merged_data, columns = cols)

In [19]:
len(mdf)

245511

In [20]:
assert len(mdf) == len(df)

if len(mdf) == len(df):
    merged_df = merge_data(df1, mdf, prompt_tag)

In [21]:
merged_df.to_csv(OUTPUT_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [22]:
len(merged_df[merged_df["A"]=="yes"])

10607

## EVENT

In [23]:
# algne andmefail
df1 = pd.read_csv(OUTPUT_FILE, encoding="utf-8",  sep=",")

FILTER_COLS = "A".split(",")
df2 = get_rows(df1, FILTER_COLS)

df = df2.copy() #.iloc[:30]
#df = df.sample(frac=1)

prompt_tag = "E"

In [24]:
df

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,tags,timex_tag,ekilex_tag,ner_tag,A
0,449951,704444,18,otsima,NaN,el,detsember,detsembrist,"Lisaks peeti samas kinni sel hetkel sõiduautoga koormat turvanud venelane , keda keskkriminaalpolitsei otsis alates eelmise aasta detsembrist taga seoses narkokuriteoga .",|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,DATE,time,NaN,no
1,11389924,18251429,3,uskuma,NaN,ad,näide,näitel,Usub enda näitel väikeriigi võimalustesse,NaN,NaN,NaN,NaN,no
2,15758867,24580070,13,täitma,NaN,ad,kinnitamine,kinnitamisel,"Ülesanded jagatakse nii , et iga ametnik täidab EAGGF arvele kantavate kulutuste kinnitamisel , väljamaksmisel või arvestamisel ainult ühte funktsiooni ning et iga ametniku tööd valvaks mõni teine ametnik .",NaN,NaN,NaN,NaN,no
3,12182310,19504428,12,jälgima,NaN,abl,internetilehekülg,internetileheküljelt,"Et teleülekannet ei olnud siin võimalik näha , jälgisin seisu Livescore'i internetileheküljelt .",NaN,NaN,NaN,NaN,no
4,53706,93122,7,vajama,NaN,in,keha,kehas,"“ Ma ei vaja plasti oma kehas selleks , et end naisena tunda ja end teistelegi naisena tutvustada .",NaN,NaN,NaN,NaN,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245506,5237903,8403209,19,märkima,NaN,in,keskus,Keskuses,""" Kõrge tööhõive ilma= sotsiaalpoliitikata on ohtlik ja ebaõiglane , "" märkis Padrig Flynn Euroopa Poliitika Uuringute Keskuses .",NaN,NaN,NaN,NaN,no
245507,2917860,4677736,14,pidama,maha,in,raadio,raadios,"Kui EHI võttis Eestis esimesena kasutusele ainepunktisüsteemi , pidasid lugupeetavad kolleegid Tartu ülikoolist raadios maha mitu pikka kõnetundi , kus nad selgitasid , et see süsteem ei hakka Eestis kunagi tööle .",NaN,NaN,NaN,NaN,no
245508,3196489,5133971,17,tormama,välja,el,koolimaja,koolimajast,"Ligi 53 tundi kestnud pantvangidraama Põhja-Osseetias Beslanis jõudis verise lahenduseni eile , kui peaaegu üheaegselt tormas koolimajast välja rühm pantvange , puhkes tulistamine ning Vene eriüksused läksid vabastusrünnakule .",|L|AL|EL|LS|LT|ALT|ELT|LST|,NaN,location,NaN,no
245509,14936649,23449513,20,puhastama,NaN,ad,ala,alal,Tallinna Vesi puhastab joogivee kvaliteedi parandamiseks veetorustikke 21. veebruaril kella 24–06 Tammsaare tee ja Laki 26 vahelisel alal .,NaN,NaN,NaN,NaN,no


In [25]:
# make task array
tasks, task_batch = task_array(df, EVENT_FEW_SHOTS, EVENT_SYSTEM_PROMPT, DEPLOYMENT)

7831it [00:00, 10184.11it/s]


In [26]:
# save task array and extra info to files

file_name = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_tasks.jsonl")

with open(file_name, 'w') as file:
    for obj in tasks:
        file.write(json.dumps(obj,ensure_ascii=False) + '\n')
        
f2 = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_data.jsonl")

with open(f2, 'w') as file:
    for obj in task_batch:
        file.write(json.dumps(obj,ensure_ascii=False) + '\n')

#### Retrieving results

In [27]:
result_file_name = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_tasks_results.jsonl")

In [28]:
# Loading data from saved file
results = []
with open(result_file_name, 'r') as file:
    for line in file:
        json_object = json.loads(line.strip())
        results.append(json_object)

In [29]:
len(results)

7831

In [30]:
merged_data = []

for res in results:
    task_id = res['custom_id']
    index = task_id.split('-')[-1]
    result = res['response']['body']['choices'][0]['message']['content']
    answers = json.loads(result)["results"]

    task_data = None
    for row in task_batch:
        if row[0] == task_id:
            task_data = row
            break
    
    if len(answers) == len(task_data[1]):
        c = 0
        for task, answer in zip(task_data[1], answers):
            if c == answer["id"]:
                d = task + (answer["a"],)
                merged_data.append(d)
                c+= 1
            else:
                print(f"missing an answer for task {task_id} id {c}")
                d = task + ("",)
                merged_data.append(d)
    else:
        print(f"{task_id}:", (len(answers)), "vs", (len(task_data[1])))
        
    if task_data is None:
        print("No task_data for task_id", task_id)
        
cols = ["sentence_id","head_id","head_loc", "verb","verb_compound","morph_case","sentence","form", prompt_tag]
mdf = pd.DataFrame(merged_data, columns = cols)

In [31]:
assert len(mdf) == len(df)

if len(mdf) == len(df):
    merged_df = merge_data(df1, mdf, prompt_tag)

In [32]:
merged_df.to_csv(OUTPUT_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [33]:
len(merged_df[merged_df["E"]=="yes"])

47360

## TIME

In [34]:
# algne andmefail
df1 = pd.read_csv(OUTPUT_FILE, encoding="utf-8",  sep=",")

FILTER_COLS = "A,E".split(",")
df2 = get_rows(df1, FILTER_COLS)

df = df2.copy() #.iloc[:30]
#df = df.sample(frac=1)

prompt_tag = "T"

In [35]:
df

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,tags,timex_tag,ekilex_tag,ner_tag,A,E
0,449951,704444,18,otsima,NaN,el,detsember,detsembrist,"Lisaks peeti samas kinni sel hetkel sõiduautoga koormat turvanud venelane , keda keskkriminaalpolitsei otsis alates eelmise aasta detsembrist taga seoses narkokuriteoga .",|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,DATE,time,NaN,no,no
1,11389924,18251429,3,uskuma,NaN,ad,näide,näitel,Usub enda näitel väikeriigi võimalustesse,NaN,NaN,NaN,NaN,no,no
3,12182310,19504428,12,jälgima,NaN,abl,internetilehekülg,internetileheküljelt,"Et teleülekannet ei olnud siin võimalik näha , jälgisin seisu Livescore'i internetileheküljelt .",NaN,NaN,NaN,NaN,no,no
4,53706,93122,7,vajama,NaN,in,keha,kehas,"“ Ma ei vaja plasti oma kehas selleks , et end naisena tunda ja end teistelegi naisena tutvustada .",NaN,NaN,NaN,NaN,no,no
5,769072,1226232,5,minema,NaN,ad,Harjumaa,Harjumaal,Politseiameti relvainstruktor läks kolmapäeval Harjumaal Loksa vallas järve ujuma ja uppus .,|L|AL|EL|LS|LT|ALT|ELT|LST|,NaN,location,LOC,no,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245506,5237903,8403209,19,märkima,NaN,in,keskus,Keskuses,""" Kõrge tööhõive ilma= sotsiaalpoliitikata on ohtlik ja ebaõiglane , "" märkis Padrig Flynn Euroopa Poliitika Uuringute Keskuses .",NaN,NaN,NaN,NaN,no,no
245507,2917860,4677736,14,pidama,maha,in,raadio,raadios,"Kui EHI võttis Eestis esimesena kasutusele ainepunktisüsteemi , pidasid lugupeetavad kolleegid Tartu ülikoolist raadios maha mitu pikka kõnetundi , kus nad selgitasid , et see süsteem ei hakka Eestis kunagi tööle .",NaN,NaN,NaN,NaN,no,no
245508,3196489,5133971,17,tormama,välja,el,koolimaja,koolimajast,"Ligi 53 tundi kestnud pantvangidraama Põhja-Osseetias Beslanis jõudis verise lahenduseni eile , kui peaaegu üheaegselt tormas koolimajast välja rühm pantvange , puhkes tulistamine ning Vene eriüksused läksid vabastusrünnakule .",|L|AL|EL|LS|LT|ALT|ELT|LST|,NaN,location,NaN,no,no
245509,14936649,23449513,20,puhastama,NaN,ad,ala,alal,Tallinna Vesi puhastab joogivee kvaliteedi parandamiseks veetorustikke 21. veebruaril kella 24–06 Tammsaare tee ja Laki 26 vahelisel alal .,NaN,NaN,NaN,NaN,no,no


In [36]:
# make task array
tasks, task_batch = task_array(df, TIME_FEW_SHOTS, TIME_SYSTEM_PROMPT, DEPLOYMENT)

6252it [00:00, 12926.95it/s]


In [37]:
# save task array and extra info to files

file_name = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_tasks.jsonl")

with open(file_name, 'w') as file:
    for obj in tasks:
        file.write(json.dumps(obj,ensure_ascii=False) + '\n')
        
f2 = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_data.jsonl")

with open(f2, 'w') as file:
    for obj in task_batch:
        file.write(json.dumps(obj,ensure_ascii=False) + '\n')

#### Retrieving results

In [38]:
result_file_name = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_tasks_results.jsonl")

In [39]:
# Loading data from saved file
results = []
with open(result_file_name, 'r') as file:
    for line in file:
        json_object = json.loads(line.strip())
        results.append(json_object)

In [40]:
len(results)

6252

In [53]:
merged_data = []

for res in results:
    task_id = res['custom_id']
    index = task_id.split('-')[-1]
    result = res['response']['body']['choices'][0]['message']['content']
    answers = json.loads(result)["results"]

    task_data = None
    for row in task_batch:
        if row[0] == task_id:
            task_data = row
            break
    
    if len(answers) == len(task_data[1]):
        c = 0
        for task, answer in zip(task_data[1], answers):
            if c == answer["id"]:
                d = task + (answer["a"],)
                merged_data.append(d)
                c+= 1
            else:
                print(f"missing an answer for task {task_id} id {c}")
                d = task + ("",)
                merged_data.append(d)
    else:
        print(f"{task_id}:", (len(answers)), "vs", (len(task_data[1])))
        expected_ids = [i for i in range(len(task_data[1]))]
        actual_ids = [a["id"] for a in answers]
        missing_ids = list(set(expected_ids) - set(actual_ids))
        #print(expected_ids)
        #print(actual_ids)
        #print(missing_ids)
        
        for i in expected_ids:
            a = ""
            for answer in answers:
                if answer["id"] == i:
                    a = answer["a"]
                    break
            for k, task in enumerate(task_data[1]):
                if k in missing_ids:
                    print(f"missing an answer for task {task_id} id {k}")
                    d = task + (a,)
                    merged_data.append(d)
                    break
                if k==i:
                    d = task + (a,)
                    merged_data.append(d)
                    break
                    
    if task_data is None:
        print("No task_data for task_id", task_id)
        
cols = ["sentence_id","head_id","head_loc", "verb","verb_compound","morph_case","sentence","form", prompt_tag]
mdf = pd.DataFrame(merged_data, columns = cols)

task-1706: 29 vs 30
missing an answer for task task-1706 id 29


In [51]:
len(mdf)

187544

In [54]:
assert len(mdf) == len(df)

if len(mdf) == len(df):
    merged_df = merge_data(df1, mdf, prompt_tag)

In [63]:
merged_df.to_csv(OUTPUT_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [64]:
len(merged_df[merged_df["T"]=="yes"])

20144

## LOCATION

In [65]:
# algne andmefail
df1 = pd.read_csv(OUTPUT_FILE, encoding="utf-8",  sep=",")

FILTER_COLS = "A,E,T".split(",")
df2 = get_rows(df1, FILTER_COLS)

df = df2.copy() #.iloc[:30]
#df = df.sample(frac=1)

prompt_tag = "L1"

In [66]:
df

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,tags,timex_tag,ekilex_tag,ner_tag,A,E,T
1,11389924,18251429,3,uskuma,NaN,ad,näide,näitel,Usub enda näitel väikeriigi võimalustesse,NaN,NaN,NaN,NaN,no,no,no
3,12182310,19504428,12,jälgima,NaN,abl,internetilehekülg,internetileheküljelt,"Et teleülekannet ei olnud siin võimalik näha , jälgisin seisu Livescore'i internetileheküljelt .",NaN,NaN,NaN,NaN,no,no,no
4,53706,93122,7,vajama,NaN,in,keha,kehas,"“ Ma ei vaja plasti oma kehas selleks , et end naisena tunda ja end teistelegi naisena tutvustada .",NaN,NaN,NaN,NaN,no,no,no
5,769072,1226232,5,minema,NaN,ad,Harjumaa,Harjumaal,Politseiameti relvainstruktor läks kolmapäeval Harjumaal Loksa vallas järve ujuma ja uppus .,|L|AL|EL|LS|LT|ALT|ELT|LST|,NaN,location,LOC,no,no,no
7,18695026,28266843,8,torkama,NaN,in,eriosa,eriosas,"Kehtiva kriminaalõiguse puudused torkavadki kõige rohkem silma eriosas , mida on korduvalt muudetud ja täiendatud ning mille tõttu on tekkinud sisemisi vastuolusid .",NaN,NaN,NaN,NaN,no,no,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245506,5237903,8403209,19,märkima,NaN,in,keskus,Keskuses,""" Kõrge tööhõive ilma= sotsiaalpoliitikata on ohtlik ja ebaõiglane , "" märkis Padrig Flynn Euroopa Poliitika Uuringute Keskuses .",NaN,NaN,NaN,NaN,no,no,no
245507,2917860,4677736,14,pidama,maha,in,raadio,raadios,"Kui EHI võttis Eestis esimesena kasutusele ainepunktisüsteemi , pidasid lugupeetavad kolleegid Tartu ülikoolist raadios maha mitu pikka kõnetundi , kus nad selgitasid , et see süsteem ei hakka Eestis kunagi tööle .",NaN,NaN,NaN,NaN,no,no,no
245508,3196489,5133971,17,tormama,välja,el,koolimaja,koolimajast,"Ligi 53 tundi kestnud pantvangidraama Põhja-Osseetias Beslanis jõudis verise lahenduseni eile , kui peaaegu üheaegselt tormas koolimajast välja rühm pantvange , puhkes tulistamine ning Vene eriüksused läksid vabastusrünnakule .",|L|AL|EL|LS|LT|ALT|ELT|LST|,NaN,location,NaN,no,no,no
245509,14936649,23449513,20,puhastama,NaN,ad,ala,alal,Tallinna Vesi puhastab joogivee kvaliteedi parandamiseks veetorustikke 21. veebruaril kella 24–06 Tammsaare tee ja Laki 26 vahelisel alal .,NaN,NaN,NaN,NaN,no,no,no


In [67]:
# make task array
tasks, task_batch = task_array(df, LOC_FEW_SHOTS, LOC_SYSTEM_PROMPT, DEPLOYMENT)

5580it [00:01, 5219.77it/s]


In [68]:
# save task array and extra info to files

file_name = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_tasks.jsonl")

with open(file_name, 'w') as file:
    for obj in tasks:
        file.write(json.dumps(obj,ensure_ascii=False) + '\n')
        
f2 = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_data.jsonl")

with open(f2, 'w') as file:
    for obj in task_batch:
        file.write(json.dumps(obj,ensure_ascii=False) + '\n')

#### Retrieving results

In [69]:
result_file_name = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_tasks_results.jsonl")

In [70]:
# Loading data from saved file
results = []
with open(result_file_name, 'r') as file:
    for line in file:
        json_object = json.loads(line.strip())
        results.append(json_object)

In [71]:
len(results)

5580

In [72]:
merged_data = []

for res in results:
    task_id = res['custom_id']
    index = task_id.split('-')[-1]
    result = res['response']['body']['choices'][0]['message']['content']
    answers = json.loads(result)["results"]

    task_data = None
    for row in task_batch:
        if row[0] == task_id:
            task_data = row
            break
    
    if len(answers) == len(task_data[1]):
        c = 0
        for task, answer in zip(task_data[1], answers):
            if c == answer["id"]:
                d = task + (answer["a"],)
                merged_data.append(d)
                c+= 1
            else:
                print(f"missing an answer for task {task_id} id {c}")
                d = task + ("",)
                merged_data.append(d)
    else:
        print(f"{task_id}:", (len(answers)), "vs", (len(task_data[1])))
        
    if task_data is None:
        print("No task_data for task_id", task_id)
        
cols = ["sentence_id","head_id","head_loc", "verb","verb_compound","morph_case","sentence","form", prompt_tag]
mdf = pd.DataFrame(merged_data, columns = cols)

In [73]:
assert len(mdf) == len(df)

if len(mdf) == len(df):
    merged_df = merge_data(df1, mdf, prompt_tag)

In [74]:
merged_df.to_csv(OUTPUT_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [75]:
len(merged_df[merged_df["L1"]=="yes"])

113657

## ABSTRACT LOCATION

In [76]:
# algne andmefail
df1 = pd.read_csv(OUTPUT_FILE, encoding="utf-8",  sep=",")

FILTER_COLS = "A,E,T,L1".split(",")
df2 = get_rows(df1, FILTER_COLS)

df = df2.copy() #.iloc[:30]
#df = df.sample(frac=1)

prompt_tag = "L2"

In [77]:
df

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,tags,timex_tag,ekilex_tag,ner_tag,A,E,T,L1
1,11389924,18251429,3,uskuma,NaN,ad,näide,näitel,Usub enda näitel väikeriigi võimalustesse,NaN,NaN,NaN,NaN,no,no,no,no
7,18695026,28266843,8,torkama,NaN,in,eriosa,eriosas,"Kehtiva kriminaalõiguse puudused torkavadki kõige rohkem silma eriosas , mida on korduvalt muudetud ja täiendatud ning mille tõttu on tekkinud sisemisi vastuolusid .",NaN,NaN,NaN,NaN,no,no,no,no
12,7403729,11884676,7,mööduma,NaN,ad,normaalkiirus,normaalkiirusel,Jan vaatas küljeaknasse -- kõik möödus normaalkiirusel .,NaN,NaN,NaN,NaN,no,no,no,no
17,16941868,26039776,1,toonitama,NaN,in,kihutuskõne,Kihutuskõnes,"Kihutuskõnes toonitas Soots liikmetega suhtlemise parandamist , mille ühe võimalusena nägi ta põllumeeste päeva pidamist , mis viimased kaks aastat ära on jäänud .",NaN,NaN,NaN,NaN,no,no,no,no
22,10238958,16430207,4,lahkuma,NaN,all,kõhuvalu,kõhuvalule,” Vaatamata ootamatule kõhuvalule lahkus Alsgaard Eestist meeldivate mälestustega .,|S|AS|LS|ST|AST|LST|,NaN,state,NaN,no,no,no,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245474,11529426,18464120,22,püüdma,NaN,in,krool,kroolis,"Istanbulis sukeldub tosin aastat tippsportlase muresid ja rõõme maitsnud Sei vette kindlate sihtidega : "" Püüan üht Eesti rekordit , soovitavalt kroolis - sel juhul alistuks ka olümpianorm 23 , 20 .",NaN,NaN,NaN,NaN,no,no,no,no
245478,12826579,20518159,2,kajama,NaN,in,pöördumine,pöördumises,Kaplinski pöördumises kajavad negatiivsed emotsioonid olgu ideede või klassitunnetuse pinnalt on harva edasiviivad .,NaN,NaN,NaN,NaN,no,no,no,no
245490,5353405,8587471,18,lammutama,NaN,ad,paber,paberil,"Kuigi Sirje oli näiteks köögi ja toa vahelise seina lõhkumise vastu , lammutasid Kätlin ja Pille selle paberil ikkagi , näidates , kui palju lisavalgust sel juhul tuppa voogaks .",NaN,NaN,NaN,NaN,no,no,no,no
245503,6298888,10119754,16,tulema,juurde,ad,hulk,hulgal,"Mõtlen siin liikuvatele sõidukitele , mida on ainuüksi viimase kümne aasta jooksul Eesti teedele kohutaval hulgal juurde tulnud .",NaN,NaN,NaN,NaN,no,no,no,no


In [78]:
# make task array
tasks, task_batch = task_array(df, ABSTRACT_FEW_SHOTS, ABSTRACT_SYSTEM_PROMPT, DEPLOYMENT)

1792it [00:00, 9539.30it/s]


In [79]:
# save task array and extra info to files

file_name = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_tasks.jsonl")

with open(file_name, 'w') as file:
    for obj in tasks:
        file.write(json.dumps(obj,ensure_ascii=False) + '\n')
        
f2 = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_data.jsonl")

with open(f2, 'w') as file:
    for obj in task_batch:
        file.write(json.dumps(obj,ensure_ascii=False) + '\n')

#### Retrieving results

In [80]:
result_file_name = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_tasks_results.jsonl")

In [81]:
# Loading data from saved file
results = []
with open(result_file_name, 'r') as file:
    for line in file:
        json_object = json.loads(line.strip())
        results.append(json_object)

In [82]:
len(results)

1792

In [83]:
merged_data = []

for res in results:
    task_id = res['custom_id']
    index = task_id.split('-')[-1]
    result = res['response']['body']['choices'][0]['message']['content']
    answers = json.loads(result)["results"]

    task_data = None
    for row in task_batch:
        if row[0] == task_id:
            task_data = row
            break
    
    if len(answers) == len(task_data[1]):
        c = 0
        for task, answer in zip(task_data[1], answers):
            if c == answer["id"]:
                d = task + (answer["a"],)
                merged_data.append(d)
                c+= 1
            else:
                print(f"missing an answer for task {task_id} id {c}")
                d = task + ("",)
                merged_data.append(d)
    else:
        print(f"{task_id}:", (len(answers)), "vs", (len(task_data[1])))
        
    if task_data is None:
        print("No task_data for task_id", task_id)
        
cols = ["sentence_id","head_id","head_loc", "verb","verb_compound","morph_case","sentence","form", prompt_tag]
mdf = pd.DataFrame(merged_data, columns = cols)

In [84]:
assert len(mdf) == len(df)

if len(mdf) == len(df):
    merged_df = merge_data(df1, mdf, prompt_tag)

In [85]:
merged_df.to_csv(OUTPUT_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [86]:
### merge L1 (location) and L2 (abstract location) into one column "L"

df1 = pd.read_csv(OUTPUT_FILE, encoding="utf-8",  sep=",")

df1['L'] = ((df1['L1'] == 'yes') | (df1['L2'] == 'yes')).map({True: 'yes', False: 'no'})
#df1['L'] = df1[['L1', 'L2']].eq('yes').any(axis=1).map({True: 'yes', False: 'no'})

df1.to_csv(OUTPUT_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [87]:
len(merged_df[merged_df["L2"]=="yes"])

14669

## STATE

In [88]:
# algne andmefail
df1 = pd.read_csv(OUTPUT_FILE, encoding="utf-8",  sep=",")

FILTER_COLS = "A,E,T,L1,L2,L".split(",")
df2 = get_rows(df1, FILTER_COLS)

df = df2.copy() #.iloc[:30]
#df = df.sample(frac=1)

prompt_tag = "S"

In [89]:
df

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,tags,timex_tag,ekilex_tag,ner_tag,A,E,T,L1,L2,L
1,11389924,18251429,3,uskuma,NaN,ad,näide,näitel,Usub enda näitel väikeriigi võimalustesse,NaN,NaN,NaN,NaN,no,no,no,no,no,no
12,7403729,11884676,7,mööduma,NaN,ad,normaalkiirus,normaalkiirusel,Jan vaatas küljeaknasse -- kõik möödus normaalkiirusel .,NaN,NaN,NaN,NaN,no,no,no,no,no,no
22,10238958,16430207,4,lahkuma,NaN,all,kõhuvalu,kõhuvalule,” Vaatamata ootamatule kõhuvalule lahkus Alsgaard Eestist meeldivate mälestustega .,|S|AS|LS|ST|AST|LST|,NaN,state,NaN,no,no,no,no,no,no
26,2678214,4296484,14,puhastama,NaN,ad,nikerdus,nikerdusel,""" Kõige ilusamad hetked olid need , kui puhastad ja puhastad mingit detaili nikerdusel , oled juba päris kindel , et see võib olla mingi liigne mustus või värvilaik , mis niikuinii kohe küljest pudeneb ja siis lõpuks tuleb saasta alt päris ehe säilinud kuldornament välja , "" kirjeldab restauraator protsessi .",NaN,NaN,NaN,NaN,no,no,no,no,no,no
27,1346144,2140210,8,sattuma,NaN,ill,hüsteerika,hüsteerikasse,""" Kuke väitel läheb eraldusruumi vaja ka hüsteerikasse sattunud kasvandike eraldamiseks - kooli tulevad lapsed on asotsiaalsetest peredest , kus laste närvisüsteem on häiritud .",|S|AS|LS|ST|AST|LST|,NaN,state,NaN,no,no,no,no,no,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245469,959456,1528524,1,hakkama,NaN,el,ahnus,Ahnusest,Ahnusest või rahanappusest on ka politseinikud ärandatud autosid omanikule tagasi müüma hakanud .,NaN,NaN,NaN,NaN,no,no,no,no,no,no
245471,12389088,19833513,19,piinama,NaN,ad,komme,kombel,"Culkin on seal väike armas poiss , kes viskab kahele röövlile telliseid pähe ja piinab neid kõige julmemal kombel , ise naeratades .",NaN,NaN,NaN,NaN,no,no,no,no,no,no
245474,11529426,18464120,22,püüdma,NaN,in,krool,kroolis,"Istanbulis sukeldub tosin aastat tippsportlase muresid ja rõõme maitsnud Sei vette kindlate sihtidega : "" Püüan üht Eesti rekordit , soovitavalt kroolis - sel juhul alistuks ka olümpianorm 23 , 20 .",NaN,NaN,NaN,NaN,no,no,no,no,no,no
245503,6298888,10119754,16,tulema,juurde,ad,hulk,hulgal,"Mõtlen siin liikuvatele sõidukitele , mida on ainuüksi viimase kümne aasta jooksul Eesti teedele kohutaval hulgal juurde tulnud .",NaN,NaN,NaN,NaN,no,no,no,no,no,no


In [90]:
# make task array
tasks, task_batch = task_array(df, STATE_FEW_SHOTS, STATE_SYSTEM_PROMPT, DEPLOYMENT)

1303it [00:00, 10411.79it/s]


In [91]:
# save task array and extra info to files

file_name = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_tasks.jsonl")

with open(file_name, 'w') as file:
    for obj in tasks:
        file.write(json.dumps(obj,ensure_ascii=False) + '\n')
        
f2 = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_data.jsonl")

with open(f2, 'w') as file:
    for obj in task_batch:
        file.write(json.dumps(obj,ensure_ascii=False) + '\n')

#### Retrieving results

In [92]:
result_file_name = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_tasks_results.jsonl")

In [93]:
# Loading data from saved file
results = []
with open(result_file_name, 'r') as file:
    for line in file:
        json_object = json.loads(line.strip())
        results.append(json_object)

In [94]:
len(results)

1303

In [95]:
merged_data = []

for res in results:
    task_id = res['custom_id']
    index = task_id.split('-')[-1]
    result = res['response']['body']['choices'][0]['message']['content']
    answers = json.loads(result)["results"]

    task_data = None
    for row in task_batch:
        if row[0] == task_id:
            task_data = row
            break
    
    if len(answers) == len(task_data[1]):
        c = 0
        for task, answer in zip(task_data[1], answers):
            if c == answer["id"]:
                d = task + (answer["a"],)
                merged_data.append(d)
                c+= 1
            else:
                print(f"missing an answer for task {task_id} id {c}")
                d = task + ("",)
                merged_data.append(d)
    else:
        print(f"{task_id}:", (len(answers)), "vs", (len(task_data[1])))
        
    if task_data is None:
        print("No task_data for task_id", task_id)
        
cols = ["sentence_id","head_id","head_loc", "verb","verb_compound","morph_case","sentence","form", prompt_tag]
mdf = pd.DataFrame(merged_data, columns = cols)

In [96]:
assert len(mdf) == len(df)

if len(mdf) == len(df):
    merged_df = merge_data(df1, mdf, prompt_tag)

In [97]:
merged_df.to_csv(OUTPUT_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [98]:
len(merged_df[merged_df["S"]=="yes"])

7923

# Create new final tag column

In [99]:

# change if needed

def new_class(row):
    mapping = {"alive":"A", "PER": "A", "location":"L", "LOC":"L", "time":"T", "event":"E", "state":"S"}

    if row["L"] == "yes":
        return "L"
    
    # kui on aeg -> "yes" 
    if row["T"] == "yes":
        return "T"
    
    # event -> "yes" 
    if row["E"] == "yes":
        return "E"
    
    # elus -> "yes" 
    if row["A"] == "yes":
        return "A"

    if row["S"] == "yes":
        return "S"
    
    if str(row["ner_tag"]) != "" and str(row["ner_tag"]) != "nan":
        if row["ner_tag"] == "ORG":
            if row["morph_case"] in ["adit", "in", "ill", "el"]: #sisekohakäänetes
                return "L"
            else: # väliskohakäänetes
                return "A"
        else:
            return mapping[row["ner_tag"]]
    
    if str(row["timex_tag"]) != "" and str(row["timex_tag"]) != "nan":
        return "T"
    
    # kui oli "no", NaN ja/või alive/time/abstract kõik olid "no"
    else:
        return ""

In [100]:
df = pd.read_csv(OUTPUT_FILE, encoding="utf-8",  sep=",")

In [101]:
df["tag"] = df.apply(new_class, axis=1)

In [102]:
df.to_csv(OUTPUT_FILE, encoding="utf-8", index=False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [103]:
df["tag"].value_counts()

tag
L    128755
E     47360
      30411
T     20258
A     10804
S      7923
Name: count, Length: 6, dtype: int64